In [1]:
from pyspark.sql.functions import max, min, avg, count, round

In [ ]:
catalog = dbutils.widgets.get("catalog")  # Dynamically resolves to citibike_dev/test/prod, depending on which target this job was deployed to

In [ ]:
# Read from the Silver table using the dynamic catalog variable,
# so this notebook works correctly regardless of which environment (dev/test/prod) it runs in
df = spark.read.table(f"{catalog}.02_silver.jc_citibike")

In [3]:
# Aggregate Silver-layer data into daily summary statistics — one row per trip_start_date.
# round(..., 2) keeps duration metrics to 2 decimal places for cleaner reporting in Power BI
df = df.groupBy("trip_start_date").agg(
    round(max("trip_duration_mins"), 2).alias("max_trip_duration_mins"),
    round(min("trip_duration_mins"), 2).alias("min_trip_duration_mins"),
    round(avg("trip_duration_mins"), 2).alias("avg_trip_duration_mins"),
    count("ride_id").alias("total_trips")
)

In [ ]:
# Write the first Gold-layer aggregation (overall daily summary) as a managed Delta table,
# using the dynamic "catalog" variable
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"{catalog}.03_gold.daily_ride_summary")